# تحلیل اکتشافی داده های اضطراب اجتماعی

**فاز دوم: EDA**

اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف

## 1. کتابخانه ها و تنظیمات

از پایتون 3.13 استفاده کنید و ورژن های زیر

In [ ]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy
import statsmodels
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_white"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"

## 2. داده های اولیه


In [ ]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

In [ ]:
print(df.duplicated().sum())

##  مشاهدات اولیه

- سطر هامون 2030 تاس و ستون هامون 22 تاس
- 9 تا ستون با داده گمشده داریم
 - که Therapy History با 90 درصد داده گمشده بدترینه
- Sleep Hours، Physical Activity و Alcohol مقدار منفی دارند
- Stress Level مقدار 15 دارد که منطقی نیست
- دوتا ستون Target و is_Anxious تغریبا یکسانند
- ستون Heart Rate و Caffeine مقدار خارج از بازه دارند

## 3. تحلیل و پاک سازی داده

### 3.1 جمعیت شناختی: Age, Gender, Occupation

### 3.2 سبک زندگی: Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality

In [ ]:
#==================
# Numeric Data
#==================
Sleep_Hours = df['Sleep Hours']
Physical_Activity = df['Physical Activity (hrs/week)']
Caffeine_Intake = df['Caffeine Intake (mg/day)']
Alcohol_Consumption = df['Alcohol Consumption (drinks/week)']


# =========================
# Categorical Data
# =========================
Smoking = df['Smoking']
Diet_Quality = df['Diet Quality (1-10)']

In [ ]:
# =========================
# Plot Numerical Data
# =========================
fig, axes = plt.subplots(4, 1, figsize=(12, 22))

# --- 1. Sleep Hours ---
axes[0].hist(
    Sleep_Hours,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0].set_title('Distribution of Sleep Hours', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sleep Hours', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2. Physical Activity ---
axes[1].hist(
    Physical_Activity,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1].set_title('Distribution of Physical Activity', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Physical Activity (hrs/week)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# --- 3. Caffeine Intake ---
axes[2].hist(
    Caffeine_Intake,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[2].set_title('Distribution of Caffeine Intake', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Caffeine Intake (mg/day)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].grid(axis='y', alpha=0.3)


# --- 4. Alcohol Consumption ---
axes[3].hist(
    Alcohol_Consumption,
    bins=10,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[3].set_title('Distribution of Alcohol Consumption', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Alcohol Consumption (drinks/week)', fontsize=11)
axes[3].set_ylabel('Frequency', fontsize=11)
axes[3].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# =========================
# Plot categorical Data
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 1.Smoking ---
smoking_counts = Smoking.value_counts()

axes[0].bar(
    smoking_counts.index.astype(str),
    smoking_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[0].set_title('Smoking Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Smoking', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2.Diet Quality ---
diet_counts = Diet_Quality.value_counts().sort_index()

axes[1].bar(
    diet_counts.index.astype(str),
    diet_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[1].set_title('Diet Quality Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# remove invlid data from Numeric columns 
# =========================

df.loc[df['Sleep Hours'] < 0, 'Sleep Hours'] = np.nan

df.loc[df['Physical Activity (hrs/week)'] < 0,
       'Physical Activity (hrs/week)'] = np.nan

df.loc[df['Alcohol Consumption (drinks/week)'] < 0,
       'Alcohol Consumption (drinks/week)'] = np.nan

In [ ]:

numeric_cols = [
    'Age',
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]

# Calculate median and replace missing values
for col in numeric_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

In [ ]:
smoking_mode = Smoking.mode()[0]
print('Smoking Mode:', smoking_mode)
df['Smoking'] = df['Smoking'].fillna(smoking_mode)

Diet_Quality_mode = Diet_Quality.mode()[0]
print('Diet Quality:', Diet_Quality_mode)
df['Diet Quality (1-10)'] = df['Diet Quality (1-10)'].fillna(Diet_Quality_mode)

### 3.3 فیزیولوژیک: Heart Rate, Breathing Rate, Sweating Level, Dizziness

### 3.4 درمان، سابقه و متغیر هدف: Family History, Medication, Therapy Sessions, Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target

### 3.5 ادغام پاک سازی ها و ساخت clean_df

## 4. ویژوال تک متغیره

### 4.1 جمعیت شناختی

### 4.2 سبک زندگی

### 4.3 فیزیولوژیک

### 4.4 درمان، سابقه و متغیر هدف

## 5. ویژوال دومتغیره

### 5.1 ستون های دسته ای با اضطراب

### 5.2 ستون های عددی با اضطراب

## 6. آزمون های آماری

### 6.1 آزمون همبستگی: عددی با عددی

### 6.2 آزمون t-test و ANOVA: دسته ای با عددی

### 6.3 آزمون chi-square: دسته ای با دسته ای

### 6.4 امتیازی: بازه های اطمینان

## 7. KPI و فیچرهای تعاملی

### 7.1 استخراج KPI و فیچرهای تعاملی از ستون ها

### 7.2 ویژوال KPI و فیچرهای جدید

## 8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر

## 9. امتیازی: ویژوال سه متغیره و بیشتر